AI agents how to interact with them using llm

In [1]:
from google import genai
from google.genai import types
import os
import requests

In [16]:
client = genai.Client(api_key=os.getenv("API_KEY"))

In [3]:
def getSum(num1, num2):
    return num1+num2+10

In [60]:
def getPrime(num):
    return num+num

In [49]:
def getCountry(country_name):
    url = f"https://restcountries.com/v3.1/name/{country_name}"

    response = requests.get(url)
    if response.status_code!=200:
        return "Could not get the details of country"
    country_details = {
    item['tld'][0]: {  # Using the first item of the tld list as the key
        "cca2": item.get('cca2'),
        "independent": item.get('independent'),
        "unMember": item.get('unMember'),
        "capital": item.get('capital'),
        "altSpellings": item.get('altSpellings'),
        "region": item.get('region'),
        "borders": item.get('borders', []), # Default to empty list if no borders
        "area": item.get('area'),
        "continents": item.get('continents'),
        "nativeName": item.get('name', {}).get('nativeName')
    }
        for item in response.json()
    }
    return country_details

defining func declaration used by model

In [6]:

get_sum_function = {
    "name": "getSum",
    "description": "Returns sum of given two numbers",
    "parameters":{
        "type": "object",
        "properties":{
            "num1":{
                "type": "integer",
                "description": "First number for addition ex: 10"
            },
             "num2":{
                "type": "integer",
                "description": "Second number for addition ex: 12"
            }
        },
        "required": ["num1","num2"]
    }
}

In [7]:
get_prime_function = {
    "name": "getPrime",
    "description": "Returns if given number is prime no or not ex: True",
    "parameters": {
        "type": "object",
        "properties": {
            "num":{
                "type": "integer",
                "description": "Number which needs to be checked if it is prime"
            }
        },
        "required":["num"]
    }
}

In [50]:
get_country_function = {
    "name": "getCountry",
    "description": "Returns dictionary of details related to country like cca2,independent,unMember,capital,altSpellings,region,borders,area,continents,nativeNames",
    "parameters": {
        "type": "object",
        "properties": {
            "country_name":{
                "type": "string",
                "description": "Country name to be searched eg. india"
            }
        },
        "required":["country_name"]
    }
}

Configuring created tools(Functions) with LLM client

In [9]:
tools = types.Tool(function_declarations=[get_sum_function,
            get_prime_function,get_country_function])
tools_map = {'getSum': getSum, 'getPrime': getPrime,'getCountry':getCountry}
config = types.GenerateContentConfig(tools=[tools])

In [17]:
chat = client.chats.create(
    model="gemini-2.5-flash-lite",
    config=config,
)


In [ ]:
user_input = None
c = 1 
while True:
    print(f"Loop {c}")
    c += 1
    if not user_input:
        user_input = input("Enter your message")
        print(user_input)
    response = chat.send_message(user_input)
    if response.function_calls:
        function_call = response.function_calls[0]
        function_name = function_call.name
        function_args = function_call.args
        if function_name in tools_map:
            function_result = tools_map[function_name](**function_args)
            response = chat.send_message(
                types.Part.from_function_response(
                    name = function_name,
                    response={"result": function_result}
                )
            )
    else:
        user_input = None
        print(f"AI: {response.text}")
        

Creating and running agent with llama.cpp locally using openai sdk

In [43]:
from openai import OpenAI
import json

In [22]:
client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="sk-no-key-required" 
)

In [61]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "getSum",
            "description": "Calculate the sum of numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "num1": {"type": "number"},
                    "num2": {"type": "number"}
                },
                "required": ["num1", "num2"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "getPrime",
            "description": "Check if a number is prime or get prime numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "num": {"type": "integer"}
                },
                "required": ["num"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "getCountry",
            "description": "Get information about a country",
            "parameters": {
                "type": "object",
                "properties": {
                    "country_name": {"type": "string"}
                },
                "required": ["country_name"]
            }
        }
    }
]

tools_map = {'getSum': getSum, 'getPrime': getPrime, 'getCountry': getCountry}

In [ ]:
MODEL_NAME = "bartowski/Meta-Llama-3.1-8B-Instruct-GGUF:Q4_K_M"

In [57]:
messages = []
user_input = None
c = 1
while True:
    print(f"Loop {c}")
    c += 1
    
    if not user_input:
        user_input = input("Enter your message: ")
        if user_input.lower().strip() == "quit":
            break
        print(user_input)
        # Append the new user message to the history
        messages.append({"role": "user", "content": user_input})
        
    # Get response from the local model
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    # Check if the model wants to call a function
    if tool_calls:
        # Append the model's request to call a tool to history
        messages.append(response_message)
        
        # Process the tool calls requested by the model
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            # OpenAI returns arguments as a JSON string; we must parse it
            function_args = json.loads(tool_call.function.arguments)
            
            if function_name in tools_map:
                # Execute your local Python function
                function_result = tools_map[function_name](**function_args)
                
                # Append the function's result back into the chat history
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": str(function_result),
                })
        
        final_response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages
        )
        ai_response = final_response.choices[0].message.content
        messages.append({"role": "assistant", "content": ai_response})
        print(f"AI: {ai_response}")
        user_input = None
    else:
        # No function call, the model just spoke to the user
        user_input = None
        ai_response = response_message.content
        messages.append({"role": "assistant", "content": ai_response})
        print(f"AI: {ai_response}")

Loop 1
is 6 a prime no
AI: No, 6 is not a prime number. A prime number is a positive integer that is divisible only by itself and 1. In this case, 6 can be divided by 2 and 3, so it does not meet the definition of a prime number.
Loop 2


In [62]:
messages = []
user_input = None
c = 1

while True:
    print(f"Loop {c}")
    c += 1
    
    if not user_input:
        user_input = input("Enter your message: ")
        if user_input.lower().strip() == "quit":
            break
        print(user_input)
        messages.append({"role": "user", "content": user_input})
        
    # 1. Get response from the local model
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls
    
    if tool_calls:
        # FIX 1: Convert the complex SDK object into a clean dictionary 
        # so local backends don't drop or misread the tool request history.
        messages.append({
            "role": "assistant",
            "content": response_message.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                } for tc in tool_calls
            ]
        })
        
        # Process the tool calls
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            if function_name in tools_map:
                # Execute your local Python function
                function_result = tools_map[function_name](**function_args)
                
                # FIX 2: Revert back to strict JSON layout for the content block.
                # Some local parsers ignore tool roles if the content isn't valid JSON.
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": json.dumps({"result": function_result}),
                })
        
        # 2. Get the final response using the exact same configuration block.
        # Keeping 'tools' active helps the local template engine parse everything correctly.
        final_response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        ai_response = final_response.choices[0].message.content
        messages.append({"role": "assistant", "content": ai_response})
        print(f"AI: {ai_response}")
        user_input = None
        
    else:
        # No function call, the model just spoke to the user
        user_input = None
        ai_response = response_message.content
        messages.append({"role": "assistant", "content": ai_response})
        print(f"AI: {ai_response}")

Loop 1
is 6 aprime
AI: 
Loop 2
is 5 a prime
AI: The result is that 5 is a prime number and the next prime is 10.
Loop 3
is 8 a prime
AI: The result is that 8 is not a prime number and the next prime is 16.
Loop 4
my name is mahesh
AI: The result is that "Could not get the details of country" because "Mahesh" is not a valid country name.
Loop 5
